# Sentiment Analysis Data Exploration

This notebook explores sentiment data collected from social media sources.

## Objectives:
1. Load and explore the dataset
2. Visualize sentiment distribution
3. Analyze trends over time
4. Identify topic patterns


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import json

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## Load Data

Loading sample sentiment data for exploration.

In [ ]:
# Generate sample data
np.random.seed(42)

# Define sentiment labels and their weights
sentiments = pd.DataFrame({
    'label': [1, 1, 0, 0, 2],
    'weight': [35, 35, 20, 20, 10]  # positive, positive, negative, negative, neutral
})

# Generate 30 days of data
days = 30
dates = [(datetime.now() - timedelta(days=days-i)).strftime('%Y-%m-%d') for i in range(days)]

# Generate topic distribution
topics = ['Technology', 'Business', 'Entertainment', 'Sports', 'Politics', 
          'Health', 'Travel', 'Food', 'Education', 'Science']

# Create DataFrame
data = []
for i in range(days):
    date = dates[i]
    topic = np.random.choice(topics)
    
    # Add sentiment distribution with some trend
    trend = 50 + 10 * np.sin(2 * np.pi * i / 14)
    base_positive = max(30, min(70, trend))
    base_negative = max(25, min(65, 70 - trend))
    
    positive = np.random.randint(int(base_positive * 0.8), int(base_positive * 1.2))
    negative = np.random.randint(int(base_negative * 0.8), int(base_negative * 1.2))
    neutral = np.random.randint(10, 30)
    
    data.append({
        'date': date,
        'positive': positive,
        'negative': negative,
        'neutral': neutral,
        'topic': topic
    })

df = pd.DataFrame(data)

print(f"Loaded {len(df)} records covering {days} days")
print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")
print(f"\nTopic distribution:")
print(df['topic'].value_counts())

## Explore Sentiment Distribution

Visualize the overall sentiment distribution.

In [ ]:
# Calculate total counts
total_positive = df['positive'].sum()
total_negative = df['negative'].sum()
total_neutral = df['neutral'].sum()
total = total_positive + total_negative + total_neutral

print(f"Total Positive: {total_positive:,}")
print(f"Total Negative: {total_negative:,}")
print(f"Total Neutral: {total_neutral:,}")
print(f"\nTotal: {total:,}")

# Calculate percentages
positive_pct = (total_positive / total) * 100 if total > 0 else 0
negative_pct = (total_negative / total) * 100 if total > 0 else 0
neutral_pct = (total_neutral / total) * 100 if total > 0 else 0

print(f"\nPositive: {positive_pct:.1f}%")
print(f"Negative: {negative_pct:.1f}%")
print(f"Neutral: {neutral_pct:.1f}%")

In [ ]:
# Create visualization
fig = go.Figure(data=[
    go.Bar(
        x=['Positive', 'Negative', 'Neutral'],
        y=[positive_pct, negative_pct, neutral_pct],
        text=[f'{total_positive:,} ({positive_pct:.1f}%)',
               f'{total_negative:,} ({negative_pct:.1f}%)',
               f'{total_neutral:,} ({neutral_pct:.1f}%)'],
        textposition='auto',
        marker_colors=['#2ecc71', '#e74c3c', '#f39c12']
    )
])

fig.update_layout(
    title='Sentiment Distribution Overview',
    xaxis_title='Sentiment Category',
    yaxis_title='Percentage (%)',
    height=500
)

fig.show()

## Analyze Sentiment Trend Over Time

Visualize how sentiment changes over the analyzed period.

In [ ]:
# Create trend chart
fig = go.Figure()

# Add positive trend
fig.add_trace(go.Scatter(
    x=df['date'],
    y=df['positive'],
    mode='lines+markers',
    name='Positive',
    line=dict(color='#2ecc71', width=3),
    marker=dict(size=8),
    fill='tozeroy',
    fillcolor='rgba(46, 204, 113, 0.1)'
))

# Add negative trend
fig.add_trace(go.Scatter(
    x=df['date'],
    y=df['negative'],
    mode='lines+markers',
    name='Negative',
    line=dict(color='#e74c3c', width=3),
    marker=dict(size=8),
    fill='tozeroy',
    fillcolor='rgba(231, 76, 60, 0.1)'
))

fig.update_layout(
    title='Sentiment Trend Over Time',
    xaxis_title='Date',
    yaxis_title='Count',
    height=400,
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

## Topic Analysis

Analyze sentiment distribution by topic.

In [ ]:
# Calculate sentiment by topic
topic_stats = df.groupby('topic').agg({
    'positive': 'sum',
    'negative': 'sum',
    'topic': 'count'
}).rename(columns={'topic': 'total_mentions'})

# Calculate positive ratio for each topic
topic_stats['positive_ratio'] = (topic_stats['positive'] / topic_stats['total_mentions']) * 100
topic_stats['negative_ratio'] = (topic_stats['negative'] / topic_stats['total_mentions']) * 100

# Display results
print("Sentiment by Topic:")
print(topic_stats[['positive', 'negative', 'positive_ratio']].round(1).sort_values('total_mentions', ascending=False))

In [ ]:
# Create treemap
fig = px.treemap(
    df,
    path=['topic'],
    values='positive',
    title='Positive Sentiment by Topic',
    color='topic',
    color_continuous_scale='RdYlGn',
    height=600
)

fig.update_layout(
    template='plotly_white',
    margin=dict(l=0, r=0, b=0, t=30)
)

fig.show()

## Summary Statistics

Final summary of the analysis.

In [ ]:
# Create summary DataFrame
summary = pd.DataFrame({
    'Metric': ['Total Records', 'Positive Sentiment', 'Negative Sentiment', 'Neutral Sentiment',
               'Overall Positive %', 'Overall Negative %', 'Date Range'],
    'Value': [len(df),
              total_positive,
              total_negative,
              total_neutral,
              f"{positive_pct:.2f}%",
              f"{negative_pct:.2f}%",
              f"{df['date'].min()} to {df['date'].max()}"
             ]
})

display(summary)

print("\n" + "="*50)
print("SENTIMENT ANALYSIS COMPLETE")
print("="*50)
print(f"Total analyses processed: {len(df):,}")
print(f"Net sentiment: {'Positive' if positive_pct > negative_pct else 'Negative'}")
print(f"Net sentiment score: {(positive_pct - negative_pct):.2f}%")